# Enriched Music Charts Analysis — 2026-W46


In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squarify

from scipy.stats import gaussian_kde

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 100

YT_RED = "#FF0000"
YT_RED_DARK = "#CC0000"
YT_BG = "#FFFFFF"
YT_SURFACE = "#F9F9F9"
YT_TEXT = "#0F0F0F"
YT_GRAY = "#606060"
YT_GRID = "#E5E5E5"


def format_number(x):
    if pd.isna(x):
        return "N/A"
    if x >= 1_000_000_000:
        return f"{x / 1_000_000_000:.1f}B"
    if x >= 1_000_000:
        return f"{x / 1_000_000:.1f}M"
    if x >= 1_000:
        return f"{x / 1_000:.1f}K"
    return f"{x:,.0f}"


# Metric functions: try to import from the package, fall back to inline.
try:
    from core.metrics import (
        collaborations,
        general_stats,
        genre_stats,
        top_countries,
        top_countries_by_likes,
        top_songs,
        top_songs_by_country,
        video_metrics,
    )
except ImportError:
    # Self-contained fallback so the notebook runs when moved or opened
    # outside the repository (e.g. Colab).
    def general_stats(df):
        return {
            "total_songs": int(len(df)),
            "unique_countries": int(df["artist_country"].nunique()),
            "unique_genres": int(df["macro_genre"].nunique()),
            "total_views": int(df["views"].sum()),
            "total_likes": int(df["likes"].sum()),
            "total_comments": int(df["comment_count"].sum()),
            "avg_views": float(df["views"].mean()),
            "avg_likes": float(df["likes"].mean()),
            "avg_engagement": float(df["engagement"].mean()),
        }

    def top_countries(df, n=10):
        return (
            df.groupby("artist_country")
            .agg(total_songs=("rank", "count"), total_views=("views", "sum"))
            .reset_index()
            .sort_values("total_songs", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )

    def top_countries_by_likes(df, n=10):
        return (
            df.groupby("artist_country")["likes"]
            .sum()
            .reset_index()
            .rename(columns={"likes": "total_likes"})
            .sort_values("total_likes", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )

    def top_songs_by_country(df, country, metric="views", n=5):
        cols = ["track_name", "artist_names", "views", "likes", "engagement"]
        return (
            df[df["artist_country"] == country]
            .nlargest(n, metric)[cols]
            .reset_index(drop=True)
        )

    def genre_stats(df):
        out = (
            df.groupby("macro_genre")
            .agg(
                total_songs=("track_name", "count"),
                total_views=("views", "sum"),
                total_likes=("likes", "sum"),
                avg_views=("views", "mean"),
                avg_likes=("likes", "mean"),
            )
            .reset_index()
        )
        out["engagement_rate"] = (
            out["total_likes"] / out["total_views"].replace(0, pd.NA) * 100
        ).fillna(0).round(2)
        return out.sort_values("total_songs", ascending=False).reset_index(drop=True)

    def video_metrics(df):
        flags = {
            "Official": "is_official_video",
            "Lyric": "is_lyric_video",
            "Live": "is_live_performance",
        }
        rows = []
        total = len(df)
        for label, flag in flags.items():
            subset = df[df[flag] == 1]
            rows.append({
                "video_type": label,
                "count": int(len(subset)),
                "pct": float(len(subset) / total * 100) if total else 0.0,
                "avg_views": float(subset["views"].mean()) if len(subset) else 0.0,
                "avg_engagement": float(subset["engagement"].mean()) if len(subset) else 0.0,
            })
        return pd.DataFrame(rows)

    def top_songs(df, metric="views", n=10):
        cols = ["rank", "track_name", "artist_names", metric, "artist_country"]
        return df.nlargest(n, metric)[cols].reset_index(drop=True)

    def collaborations(df):
        grouped = (
            df.groupby("is_collaboration")
            .agg(
                count=("track_name", "count"),
                avg_views=("views", "mean"),
                avg_engagement=("engagement", "mean"),
            )
            .reset_index()
        )
        grouped["is_collaboration"] = grouped["is_collaboration"].map(
            {0: "Solo", 1: "Collaboration"}
        )
        return grouped


DB_PATH = Path("../../charts_archive/3_enrich-chart-data/youtube_charts_2026-W46_enriched.db")
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM enriched_songs", conn)
conn.close()

df["upload_date"] = pd.to_datetime(df["upload_date"], errors="coerce")
df["upload_quarter"] = df["upload_date"].dt.quarter
df["engagement"] = np.where(
    df["views"] > 0, (df["likes"] / df["views"] * 100).round(2), 0.0
)


## 1. Introduction

**Week:** 2026-W46 | **Period:** 2026-11-09 → 2026-11-15 | **Days:** 7

## 2. General Statistics

In [ ]:
stats = general_stats(df)
display(pd.DataFrame([stats]))

## 3. Country Analysis

In [ ]:
# Top 10 countries by song count
top_countries(df, n=10)

In [ ]:
# Top 10 countries by total likes
top_countries_by_likes(df, n=10)

In [ ]:
# Top 5 songs by country (top 10 countries)
for country in df['artist_country'].value_counts().head(10).index:
    print(f'\n{country}:')
    display(top_songs_by_country(df, country, metric='views', n=5))

## 4. Genre Analysis

In [ ]:
# Genre distribution treemap (top 15 genres)
gs = genre_stats(df).head(15)
sizes = gs['total_songs'].values
labels = [
    f"{g}\n{format_number(s)}"
    for g, s in zip(gs['macro_genre'], gs['total_songs'])
]
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(sizes)))
fig, ax = plt.subplots(figsize=(14, 8))
squarify.plot(
    sizes=sizes, label=labels, alpha=0.9, color=colors,
    text_kwargs={'fontsize': 10, 'fontweight': 'bold', 'color': 'white'},
    ax=ax,
)
ax.set_title('Genre Distribution by Song Count', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Engagement rate by genre (top 10)
gs = genre_stats(df).sort_values('engagement_rate', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Reds(np.linspace(0.4, 1, len(gs)))[::-1]
ax.barh(gs['macro_genre'], gs['engagement_rate'], color=colors, height=0.65)
ax.invert_yaxis()
ax.set_xlabel('Engagement Rate (%)')
ax.set_title('Top 10 Genres by Engagement Rate', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Video Metrics

In [ ]:
vm = video_metrics(df)
display(vm)

In [ ]:
# Average views by video type
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=vm, x='video_type', y='avg_views', ax=ax, color='#FC4B4C')
ax.set_title('Average Views by Video Type', fontweight='bold')
ax.set_ylabel('Average Views')
ax.set_xlabel('Video Type')
plt.tight_layout()
plt.show()

In [ ]:
# Average engagement by video type
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=vm, x='video_type', y='avg_engagement', ax=ax, color='#FC4B4C')
ax.set_title('Average Engagement by Video Type', fontweight='bold')
ax.set_ylabel('Engagement (%)')
ax.set_xlabel('Video Type')
plt.tight_layout()
plt.show()

## 6. Song Metrics

In [ ]:
# Top 10 songs by views
display(top_songs(df, metric='views', n=10))

In [ ]:
# Top 10 songs by likes
display(top_songs(df, metric='likes', n=10))

In [ ]:
# Top 10 songs by engagement
display(top_songs(df, metric='engagement', n=10))

## 7. Collaborations Analysis

In [ ]:
display(collaborations(df))

In [ ]:
# Views vs number of artists
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.scatterplot(
    data=df, x='artist_count', y='views', hue='is_collaboration',
    palette={0: '#282828', 1: 'red'}, ax=axes[0], alpha=0.6,
)
axes[0].set_title('Views vs Number of Artists', fontweight='bold')
sns.scatterplot(
    data=df, x='artist_count', y='engagement', hue='is_collaboration',
    palette={0: '#282828', 1: 'red'}, ax=axes[1], alpha=0.6,
)
axes[1].set_title('Engagement vs Number of Artists', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Executive Summary

In [ ]:
stats = general_stats(df)
print(f"Songs: {stats['total_songs']}")
print(f"Countries: {stats['unique_countries']}")
print(f"Genres: {stats['unique_genres']}")
print(f"Total views: {format_number(stats['total_views'])}")
print(f"Total likes: {format_number(stats['total_likes'])}")
print(f"Avg engagement: {stats['avg_engagement']:.2f}%")